In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast, split, lit

spark = SparkSession.builder \
  .appName("IcebergTableManagement") \
  .config("spark.executor.memory", "4g") \
  .config("spark.driver.memory", "4g") \
  .config("spark.sql.shuffle.partitions", "200") \
  .config("spark.sql.files.maxPartitionBytes", "134217728") \
  .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
  .config("spark.dynamicAllocation.enabled", "true") \
  .config("spark.dynamicAllocation.minExecutors", "1") \
  .config("spark.dynamicAllocation.maxExecutors", "50") \
  .getOrCreate()


24/12/15 15:01:31 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
spark.sql('show databases').show()

+---------+
|namespace|
+---------+
+---------+



In [3]:
spark.sql('CREATE DATABASE homework')

DataFrame[]

In [4]:
spark.sql('show tables in homework').show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [5]:
matches = spark.read.option("header", "true") \
                        .option("inferSchema", "true") \
                        .csv("/home/iceberg/data/matches.csv") \

matchDetails = spark.read.option("header", "true") \
                        .option("inferSchema", "true") \
                        .csv("/home/iceberg/data/match_details.csv") \

medalsMatchesPlayers = spark.read.option("header", "true") \
                        .option("inferSchema", "true") \
                        .csv("/home/iceberg/data/medals_matches_players.csv") \

maps =  spark.read.option("header", "true") \
                        .option("inferSchema", "true") \
                        .csv("/home/iceberg/data/maps.csv") \

medals =  spark.read.option("header", "true") \
                        .option("inferSchema", "true") \
                        .csv("/home/iceberg/data/medals.csv")

In [6]:
matchDetails.columns

['match_id',
 'player_gamertag',
 'previous_spartan_rank',
 'spartan_rank',
 'previous_total_xp',
 'total_xp',
 'previous_csr_tier',
 'previous_csr_designation',
 'previous_csr',
 'previous_csr_percent_to_next_tier',
 'previous_csr_rank',
 'current_csr_tier',
 'current_csr_designation',
 'current_csr',
 'current_csr_percent_to_next_tier',
 'current_csr_rank',
 'player_rank_on_team',
 'player_finished',
 'player_average_life',
 'player_total_kills',
 'player_total_headshots',
 'player_total_weapon_damage',
 'player_total_shots_landed',
 'player_total_melee_kills',
 'player_total_melee_damage',
 'player_total_assassinations',
 'player_total_ground_pound_kills',
 'player_total_shoulder_bash_kills',
 'player_total_grenade_damage',
 'player_total_power_weapon_damage',
 'player_total_power_weapon_grabs',
 'player_total_deaths',
 'player_total_assists',
 'player_total_grenade_kills',
 'did_win',
 'team_id']

In [7]:
bucketedMatchDetails = """
CREATE TABLE IF NOT EXISTS homework.match_details_bucketed (
    match_id STRING,
    player_gamertag STRING,
    player_total_kills INTEGER,
    player_total_deaths INTEGER
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(bucketedMatchDetails)

DataFrame[]

In [8]:
matchDetails.select("match_id", "player_gamertag", "player_total_kills", "player_total_deaths") \
    .write.mode("append") \
    .bucketBy(16, "match_id").saveAsTable("homework.match_details_bucketed")

In [9]:
medalsMatchesPlayers.columns

['match_id', 'player_gamertag', 'medal_id', 'count']

In [10]:
bucketedMedalsMatchesPlayersDetails = """
CREATE TABLE IF NOT EXISTS homework.medals_matches_players (
    match_id STRING,
    player_gamertag STRING,
    medal_id STRING,
    count INTEGER
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(bucketedMedalsMatchesPlayersDetails)

DataFrame[]

In [11]:
medalsMatchesPlayers.select("match_id", "player_gamertag", "medal_id", "count") \
    .write.mode("append") \
    .bucketBy(16, "match_id").saveAsTable("homework.medals_matches_players")

In [12]:
spark.sql('show tables in homework').show()

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
| homework|match_details_buc...|      false|
| homework|medals_matches_pl...|      false|
+---------+--------------------+-----------+



In [13]:
distinctDates = matches.select("completion_date").distinct().collect()

In [14]:
matches.columns

['match_id',
 'mapid',
 'is_team_game',
 'playlist_id',
 'game_variant_id',
 'is_match_over',
 'completion_date',
 'match_duration',
 'game_mode',
 'map_variant_id']

In [15]:
matchesBucketedDDL = """
CREATE TABLE IF NOT EXISTS homework.matches_bucketed (
    match_id STRING,
    mapid STRING,
    is_team_game BOOLEAN,
    playlist_id STRING,
    completion_date TIMESTAMP
)
USING iceberg
PARTITIONED BY (completion_date, bucket(16, match_id))
"""
spark.sql(matchesBucketedDDL)

DataFrame[]

In [16]:
from pyspark.storagelevel import StorageLevel

In [17]:
for row in distinctDates:
    date = row['completion_date']
    filteredMatches = matches.filter(matches.completion_date == date)
    optimizedMatches = filteredMatches \
        .select("match_id", "mapid", "is_team_game", "playlist_id", "completion_date") \
        .repartition(16, "match_id") \
        .persist(StorageLevel.MEMORY_AND_DISK)

    optimizedMatches.write \
    .mode("append") \
    .bucketBy(16, "match_id") \
    .partitionBy("completion_date") \
    .saveAsTable("homework.matches_bucketed")

In [19]:
matches_result = spark.sql("SELECT * FROM homework.matches_bucketed")
matches_result.show()

+--------------------+--------------------+------------+--------------------+-------------------+
|            match_id|               mapid|is_team_game|         playlist_id|    completion_date|
+--------------------+--------------------+------------+--------------------+-------------------+
|24bbd320-9bba-4aa...|cb914b9e-f206-11e...|        true|892189e9-d712-4bd...|2016-08-16 00:00:00|
|1a85415b-23f7-43b...|cdee4e70-f206-11e...|        true|892189e9-d712-4bd...|2016-08-16 00:00:00|
|51b5be27-8183-474...|cdb934b0-f206-11e...|        true|892189e9-d712-4bd...|2016-08-16 00:00:00|
|ba94761e-d63a-403...|caacb800-f206-11e...|        true|c98949ae-60a8-43d...|2016-08-16 00:00:00|
|9be0f082-b7fa-47f...|cb914b9e-f206-11e...|        true|892189e9-d712-4bd...|2015-11-12 00:00:00|
|26b1d7e1-5149-4b4...|caacb800-f206-11e...|        true|c98949ae-60a8-43d...|2015-11-12 00:00:00|
|a666d82c-e0b6-408...|ce1dc2de-f206-11e...|        true|892189e9-d712-4bd...|2015-11-12 00:00:00|
|5092887d-41c1-4d0..

In [20]:
spark.sql('SELECT * from homework.match_details_bucketed mdb JOIN homework.matches_bucketed mb ON mdb.match_id = mb.match_id').explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [match_id#46391], [match_id#46395], Inner
   :- Sort [match_id#46391 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(match_id#46391, 200), ENSURE_REQUIREMENTS, [plan_id=16129]
   :     +- BatchScan demo.homework.match_details_bucketed[match_id#46391, player_gamertag#46392, player_total_kills#46393, player_total_deaths#46394] demo.homework.match_details_bucketed (branch=null) [filters=match_id IS NOT NULL, groupedBy=] RuntimeFilters: []
   +- Sort [match_id#46395 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(match_id#46395, 200), ENSURE_REQUIREMENTS, [plan_id=16130]
         +- BatchScan demo.homework.matches_bucketed[match_id#46395, mapid#46396, is_team_game#46397, playlist_id#46398, completion_date#46399] demo.homework.matches_bucketed (branch=null) [filters=match_id IS NOT NULL, groupedBy=] RuntimeFilters: []




In [48]:
matches.createOrReplaceTempView("matchesView")

In [49]:
matchDetails.createOrReplaceTempView("matchDetailsView")

In [51]:
spark.sql('SELECT * from matchDetailsView mdv JOIN matchesView mv ON mdv.match_id = mv.match_id').explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [match_id#72], [match_id#35], Inner
   :- Sort [match_id#72 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(match_id#72, 200), ENSURE_REQUIREMENTS, [plan_id=16290]
   :     +- Filter isnotnull(match_id#72)
   :        +- FileScan csv [match_id#72,player_gamertag#73,previous_spartan_rank#74,spartan_rank#75,previous_total_xp#76,total_xp#77,previous_csr_tier#78,previous_csr_designation#79,previous_csr#80,previous_csr_percent_to_next_tier#81,previous_csr_rank#82,current_csr_tier#83,current_csr_designation#84,current_csr#85,current_csr_percent_to_next_tier#86,current_csr_rank#87,player_rank_on_team#88,player_finished#89,player_average_life#90,player_total_kills#91,player_total_headshots#92,player_total_weapon_damage#93,player_total_shots_landed#94,player_total_melee_kills#95,... 12 more fields] Batched: false, DataFilters: [isnotnull(match_id#72)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/h

In [21]:
createBucketJoinedDDL = """
CREATE TABLE IF NOT EXISTS homework.final_matches_joined
USING iceberg
PARTITIONED BY (bucket(16, match_id))
AS
(
    SELECT mdb.match_id, mdb.player_gamertag, mdb.player_total_kills, mdb.player_total_deaths,
        mb.mapid, mb.is_team_game, mb.playlist_id, mb.completion_date,
        mmp.medal_id, mmp.count
    FROM ((homework.match_details_bucketed mdb
    INNER JOIN homework.matches_bucketed mb ON mdb.match_id = mb.match_id)
    INNER JOIN homework.medals_matches_players mmp ON mdb.match_id = mmp.match_id)
)
"""

In [22]:
spark.sql(createBucketJoinedDDL)

DataFrame[]

In [23]:
spark.sql('select count(*) from homework.final_matches_joined').show()

+--------+
|count(1)|
+--------+
| 6885858|
+--------+



In [25]:
spark.sql('select * from homework.final_matches_joined').show()

+--------------------+---------------+------------------+-------------------+--------------------+------------+--------------------+-------------------+----------+-----+
|            match_id|player_gamertag|player_total_kills|player_total_deaths|               mapid|is_team_game|         playlist_id|    completion_date|  medal_id|count|
+--------------------+---------------+------------------+-------------------+--------------------+------------+--------------------+-------------------+----------+-----+
|0377e616-bf8b-4a4...|     Snipe Envy|                14|                  8|cdee4e70-f206-11e...|        true|bc0f8ad6-31e6-4a1...|2016-04-23 00:00:00|2078758684|    2|
|0377e616-bf8b-4a4...|     Snipe Envy|                14|                  8|cdee4e70-f206-11e...|        true|bc0f8ad6-31e6-4a1...|2016-04-23 00:00:00| 848240062|    1|
|0377e616-bf8b-4a4...|     Snipe Envy|                14|                  8|cdee4e70-f206-11e...|        true|bc0f8ad6-31e6-4a1...|2016-04-23 00:00:0

In [37]:
averageKills = """
SELECT dfm.player_gamertag, 
       AVG(dfm.player_total_kills) OVER (PARTITION BY dfm.player_gamertag) AS average_kills
FROM 
    (SELECT DISTINCT match_id, player_gamertag, player_total_kills FROM homework.final_matches_joined) AS dfm
ORDER BY average_kills DESC
"""

In [38]:
spark.sql(averageKills).show()

+---------------+-------------+
|player_gamertag|average_kills|
+---------------+-------------+
|   gimpinator14|        109.0|
|  I Johann117 I|         96.0|
|BudgetLegendary|         83.0|
|      GsFurreal|         75.0|
|   Sexy is Back|         73.0|
|   killerguy789|         68.0|
|THC GUILTYSPARK|         67.0|
|    HisLattice1|         66.0|
|PrimePromethean|         66.0|
|     taurenmonk|         64.0|
|   Dinosaur B0B|         63.0|
|WhiteMountainDC|         63.0|
|        Darugis|         62.0|
|       BlightNB|         62.0|
|     MONKEYBAKE|         62.0|
|    ManicZ0mb1e|         61.0|
|Lord Leonidamir|         60.0|
|LEGENDARY link0|         60.0|
|    ohh Replxys|         60.0|
|Recon Omega2552|         59.0|
+---------------+-------------+
only showing top 20 rows



In [43]:
mostPlayedPlaylist = """
SELECT dfm.playlist_id AS id, COUNT(dfm.playlist_id) AS playlist_count
FROM
    (SELECT DISTINCT match_id, playlist_id FROM homework.final_matches_joined) AS dfm
GROUP BY id
ORDER BY playlist_count DESC
LIMIT 3
"""

In [44]:
spark.sql(mostPlayedPlaylist).show()

+--------------------+--------------+
|                  id|playlist_count|
+--------------------+--------------+
|f72e0ef0-7c4a-430...|          7640|
|2323b76a-db98-4e0...|          3171|
|892189e9-d712-4bd...|          1961|
+--------------------+--------------+



In [45]:
maps.columns

['mapid', 'name', 'description']

In [50]:
maps.createOrReplaceTempView("mapsView")

In [59]:
mostPlayedMap = """
SELECT dfm.mapid, name, COUNT(dfm.mapid) as map_count
FROM
    (SELECT DISTINCT match_id, mapid FROM homework.final_matches_joined) AS dfm
    INNER JOIN mapsView
    ON dfm.mapid = mapsView.mapid
GROUP BY dfm.mapid, name
ORDER BY map_count DESC
LIMIT 3
"""

In [60]:
spark.sql(mostPlayedMap).show()

+--------------------+--------------+---------+
|               mapid|          name|map_count|
+--------------------+--------------+---------+
|c7edbf0f-f206-11e...|Breakout Arena|     7032|
|c74c9d0f-f206-11e...|        Alpine|     1358|
|cdb934b0-f206-11e...|        Empire|     1347|
+--------------------+--------------+---------+



In [61]:
medals.columns

['medal_id',
 'sprite_uri',
 'sprite_left',
 'sprite_top',
 'sprite_sheet_width',
 'sprite_sheet_height',
 'sprite_width',
 'sprite_height',
 'classification',
 'description',
 'name',
 'difficulty']

In [62]:
medals.createOrReplaceTempView("medalsView")

In [65]:
spark.sql('select name from medalsView').show()

+--------------+
|          name|
+--------------+
|          NULL|
|          NULL|
| Buzzer Beater|
|    Vanquisher|
|Spartan Charge|
|  Ghost Assist|
|    Grunt Kill|
|       Bifecta|
|  Perfect Kill|
|  Base Defense|
| Killing Spree|
| Team Takedown|
|    Extinction|
|   Hard Target|
|     Quickdraw|
| Capture Spree|
|Big Gun Runner|
| Team Takedown|
|      Fastball|
|  Shotgun Kill|
+--------------+
only showing top 20 rows



In [66]:
mostKillingSpreeMedals = """
SELECT dfm.mapid, SUM(dfm.count) as total_killing_spree
FROM
    (SELECT DISTINCT match_id, mapid, medal_id, count FROM homework.final_matches_joined) AS dfm
        INNER JOIN medalsView
        ON dfm.medal_id = medalsView.medal_id
WHERE medalsView.name = 'Killing Spree'
GROUP BY dfm.mapid
ORDER BY total_killing_spree DESC
LIMIT 3
"""

In [67]:
spark.sql(mostKillingSpreeMedals).show()

+--------------------+-------------------+
|               mapid|total_killing_spree|
+--------------------+-------------------+
|c7edbf0f-f206-11e...|               5165|
|c74c9d0f-f206-11e...|               2710|
|c7805740-f206-11e...|               1827|
+--------------------+-------------------+



In [68]:
medals.columns

['medal_id',
 'sprite_uri',
 'sprite_left',
 'sprite_top',
 'sprite_sheet_width',
 'sprite_sheet_height',
 'sprite_width',
 'sprite_height',
 'classification',
 'description',
 'name',
 'difficulty']

In [69]:
maps.columns

['mapid', 'name', 'description']

In [70]:
medals.join(
    broadcast(maps)).show()
    

+----------+----------+-----------+----------+------------------+-------------------+------------+-------------+--------------+-----------+----+----------+--------------------+-------------------+--------------------+
|  medal_id|sprite_uri|sprite_left|sprite_top|sprite_sheet_width|sprite_sheet_height|sprite_width|sprite_height|classification|description|name|difficulty|               mapid|               name|         description|
+----------+----------+-----------+----------+------------------+-------------------+------------+-------------+--------------+-----------+----+----------+--------------------+-------------------+--------------------+
|2315448068|      NULL|       NULL|      NULL|              NULL|               NULL|        NULL|         NULL|          NULL|       NULL|NULL|      NULL|c93d708f-f206-11e...|              Urban|Andesia was the c...|
|2315448068|      NULL|       NULL|      NULL|              NULL|               NULL|        NULL|         NULL|          NULL| 

In [71]:
full_df = spark.read.table('homework.final_matches_joined')

In [72]:
full_df

DataFrame[match_id: string, player_gamertag: string, player_total_kills: int, player_total_deaths: int, mapid: string, is_team_game: boolean, playlist_id: string, completion_date: timestamp, medal_id: string, count: int]

In [73]:
full_df.sortWithinPartitions('playlist_id', ascending=False)

DataFrame[match_id: string, player_gamertag: string, player_total_kills: int, player_total_deaths: int, mapid: string, is_team_game: boolean, playlist_id: string, completion_date: timestamp, medal_id: string, count: int]

In [74]:
sort_playlist_df = full_df.sortWithinPartitions('playlist_id', ascending=False)

In [75]:
sort_playlist_df.head()

Row(match_id='bd744ad0-23d2-4fa7-921c-fc194b4243b9', player_gamertag='Oppo Taco 14', player_total_kills=8, player_total_deaths=6, mapid='cc040aa1-f206-11e4-a3e0-24be05e24f7e', is_team_game=True, playlist_id='fe2ad4e1-3def-46a9-a18e-9ab685aa66d4', completion_date=datetime.datetime(2016, 2, 6, 0, 0), medal_id='3400287617', count=3)

In [76]:
sort_playlist_df.size

AttributeError: 'DataFrame' object has no attribute 'size'

In [77]:
print((sort_playlist_df.count(), len(sort_playlist_df.columns)))

(6885858, 10)


In [78]:
print((full_df.count(), len(full_df.columns)))

(6885858, 10)


In [79]:
sort_maps_df = full_df.sortWithinPartitions('mapid', ascending=False)

In [80]:
sort_maps_df.size

AttributeError: 'DataFrame' object has no attribute 'size'

In [81]:
%%sql

SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'sorted' 
FROM homework.final_matches_joined.files

24/12/15 17:21:15 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


size,num_files,sorted
6093215,16,sorted


In [82]:
%%sql

CREATE TABLE IF NOT EXISTS homework.sort_playlist (
    match_id STRING,
    player_gamertag STRING,
    player_total_kills INTEGER,
    player_total_deaths INTEGER,
    mapid STRING,
    is_team_game BOOLEAN,
    playlist_id STRING,
    completion_date TIMESTAMP,
    medal_id STRING,
    count INTEGER
)
USING iceberg
PARTITIONED BY (playlist_id)

++
||
++
++

In [83]:
sort_playlist_df.write.mode("append").saveAsTable("homework.sort_playlist")

In [84]:
%%sql

SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'sorted' 
FROM homework.sort_playlist.files

size,num_files,sorted
10679091,23,sorted


In [85]:
%%sql

CREATE TABLE IF NOT EXISTS homework.sort_mapid (
    match_id STRING,
    player_gamertag STRING,
    player_total_kills INTEGER,
    player_total_deaths INTEGER,
    mapid STRING,
    is_team_game BOOLEAN,
    playlist_id STRING,
    completion_date TIMESTAMP,
    medal_id STRING,
    count INTEGER
)
USING iceberg
PARTITIONED BY (mapid)

++
||
++
++

In [86]:
map_id_df = full_df.sortWithinPartitions('mapid')

In [87]:
map_id_df.write.mode("append").saveAsTable("homework.sort_mapid")

In [88]:
%%sql

SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'sorted' 
FROM homework.sort_mapid.files

size,num_files,sorted
11108126,16,sorted
